# Penyelesaian Sistem Persamaan Linear 3 Variabel
**Metode:** Eliminasi Gauss dengan *partial pivoting* + substitusi mundur

**Nama:** [ISI NAMA ANDA] | **NIM:** [ISI NIM] | **Mata Kuliah:** Komputasi Numerik

Cara pakai: jalankan tiap cell berurutan dengan **Shift + Enter**.

Sistem yang diselesaikan:

2x + y - z = 8

-3x - y + 2z = -11

-2x + y + 2z = -3

## Cell 1 - Import library
NumPy dipakai untuk array/matriks dan operasi vektor. Cell ini hanya perlu dijalankan sekali.

In [ ]:
import numpy as np  # memuat NumPy dan memberi alias "np" agar penulisan singkat

## Cell 2 - Input data sistem
Sistem ditulis dalam bentuk matriks **A·x = b**. A adalah koefisien, b adalah ruas kanan.
Anda bisa mengganti angka-angkanya untuk mencoba sistem lain.

In [ ]:
A = np.array([[ 2,  1, -1],   # koefisien baris 1: 2x + 1y - 1z
              [-3, -1,  2],   # koefisien baris 2: -3x - 1y + 2z
              [-2,  1,  2]],  # koefisien baris 3: -2x + 1y + 2z
             dtype=float)     # dipaksa bertipe desimal agar pembagian tidak terpotong
b = np.array([8, -11, -3], dtype=float)  # konstanta ruas kanan tiap persamaan
print("Matriks A:\n", A)  # menampilkan matriks koefisien
print("Vektor b :", b)     # menampilkan vektor konstanta

## Cell 3 - Fungsi Eliminasi Gauss
Tahapan: (1) bentuk matriks augmented, (2) pilih pivot terbesar, (3) tukar baris, (4) eliminasi ke bawah sehingga terbentuk segitiga atas, (5) substitusi mundur.

In [ ]:
def eliminasi_gauss(A, b):  # mendefinisikan fungsi dengan input matriks A dan vektor b
    n = len(b)  # n = jumlah persamaan/variabel (di sini 3)
    M = np.hstack((A.copy(), b.reshape(-1, 1)))  # gabungkan A dan b menjadi matriks augmented [A|b]; copy agar A asli aman

    for k in range(n - 1):  # k = kolom pivot yang sedang diproses
        p = k + np.argmax(np.abs(M[k:, k]))  # cari baris dengan nilai mutlak terbesar di kolom k (partial pivoting)
        if abs(M[p, k]) < 1e-12:  # jika pivot hampir nol...
            raise ValueError("Matriks singular: solusi tunggal tidak ada")  # ...hentikan dengan pesan kesalahan
        if p != k:  # jika baris pivot bukan baris saat ini
            M[[k, p]] = M[[p, k]]  # tukar posisi baris k dan baris p
        for i in range(k + 1, n):  # proses semua baris di bawah pivot
            faktor = M[i, k] / M[k, k]  # pengali agar elemen M[i,k] menjadi nol
            M[i, k:] = M[i, k:] - faktor * M[k, k:]  # operasi baris: Ri = Ri - faktor * Rk

    if abs(M[n - 1, n - 1]) < 1e-12:  # cek pivot terakhir
        raise ValueError("Matriks singular: solusi tunggal tidak ada")  # tidak bisa disubstitusi mundur

    x = np.zeros(n)  # siapkan wadah solusi
    for i in range(n - 1, -1, -1):  # substitusi mundur: dari baris terakhir ke atas
        jumlah = np.dot(M[i, i + 1:n], x[i + 1:])  # jumlah suku variabel yang sudah diketahui
        x[i] = (M[i, n] - jumlah) / M[i, i]  # hitung variabel ke-i
    return x, M  # kembalikan solusi dan matriks segitiga atas

## Cell 4 - Menjalankan fungsi
Hasil ditampilkan bersama matriks segitiga atas agar proses eliminasi bisa diperiksa.

In [ ]:
x, segitiga = eliminasi_gauss(A, b)  # panggil fungsi, simpan solusi dan matriks hasil eliminasi
print("Matriks setelah eliminasi:\n", np.round(segitiga, 4))  # tampilkan segitiga atas (dibulatkan 4 desimal)
print()  # baris kosong pemisah
for nama, nilai in zip("xyz", x):  # pasangkan nama variabel dengan nilainya
    print(f"{nama} = {nilai:.6f}")  # cetak setiap variabel dengan 6 angka di belakang koma

## Cell 5 - Verifikasi hasil
Dua cara: (1) hitung residual A·x - b, harus mendekati 0; (2) bandingkan dengan `np.linalg.solve`.

In [ ]:
residual = A @ x - b  # substitusi solusi ke sistem asli; selisih terhadap b
print("Residual     :", residual)  # nilai seharusnya sangat kecil (~1e-15)
x_numpy = np.linalg.solve(A, b)  # solusi pembanding dari fungsi bawaan NumPy
print("Solusi NumPy :", x_numpy)  # tampilkan solusi pembanding
print("Cocok?       :", np.allclose(x, x_numpy))  # True jika kedua solusi sama dalam toleransi

## Cell 6 - Uji kasus tanpa solusi tunggal
Baris ke-3 dibuat kelipatan baris ke-1 sehingga sistem singular. Program harus menolak, bukan memberi angka salah.

In [ ]:
A_singular = np.array([[1, 2, 3],   # baris 1
                       [4, 5, 6],   # baris 2
                       [2, 4, 6]],  # baris 3 = 2 x baris 1 (bergantung linear)
                      dtype=float)  # tipe desimal
b_singular = np.array([1, 2, 3], dtype=float)  # ruas kanan sembarang
try:  # coba jalankan, tangkap kesalahan jika ada
    eliminasi_gauss(A_singular, b_singular)  # seharusnya memicu ValueError
except ValueError as e:  # bila terjadi ValueError
    print("Terdeteksi:", e)  # tampilkan pesan agar kita tahu sistem tidak punya solusi tunggal